Python for Data 26: ANOVA  back to index  One-Way ANOVA  The one-way ANOVA tests whether the mean of some numeric variable differs across the levels of one categorical variable. It essentially answers the question: do any of the group means differ from one another? We won’t get into the details of carrying out an ANOVA by hand as it involves more calculations than the t-test, but the process is similar: you go through several calculations to arrive at a test statistic and then you compare the test statistic to a critical value based on a probability distribution. In the case of the ANOVA, you use the “f-distribution”. The scipy library has a function for carrying out one-way ANOVA tests called scipy.stats.f_oneway(). Let’s generate some fake voter age and demographic data and use the ANOVA to compare average ages across the groups:  Log in to Kaggle and Download Data  Before downloading data from Kaggle, ensure you are logged in using  kagglehub.login() .  Important: Complete the interactive Kaggle login!  To resolve the   UnauthenticatedError , you must manually complete the interactive login process for   kagglehub.login() . This usually involves following a link or pasting an API token. Once you have successfully authenticated your Kaggle account, you can proceed to run the code cell below to execute  kagglehub.login()   and then the subsequent cell to download the data.  # Log in to Kaggle import   kagglehub kagglehub.login() VBox(children=(HTML(value='<center> <img\nsrc=https://www.kaggle.com/static/images/site-logo.png\nalt=\'Kaggle...  In lesson 24 we introduced the t-test for checking whether the means of two groups differ. The t-test works well when dealing with two groups, but sometimes we want to compare more than two groups at the same time. For example, if we wanted to test whether voter age differs based on some categorical variable like race, we have to compare the means of each level or group the variable. We could carry out a separate t-test for each pair of groups, but when you conduct many tests you increase the chances of false positives. The   analysis of variance   or ANOVA is a statistical inference test that lets you compare multiple groups at the same time.

--- End of Page 1 ---

import   numpy   as   np import   pandas   as   pd import   matplotlib.pyplot   as   plt import   scipy.stats   as   stats np.random.seed( 12 ) races   =   [ "asian" , "black" , "hispanic" , "other" , "white" ] # Generate random data voter_race   =   np.random.choice(a =   races, p   =   [ 0.05 ,   0.15   , 0.25 ,   0.05 ,   0.5 ], size = 1000 ) voter_age   =   stats.poisson.rvs(loc = 18 , mu = 30 , size = 1000 ) # Group age data by race voter_frame   =   pd.DataFrame({ "race" :voter_race, "age" :voter_age}) groups   =   voter_frame.groupby( "race" ).groups # Etract individual groups asian   =   voter_age[groups[ "asian" ]] black   =   voter_age[groups[ "black" ]] hispanic   =   voter_age[groups[ "hispanic" ]] other   =   voter_age[groups[ "other" ]] white   =   voter_age[groups[ "white" ]] # Perform the ANOVA stats.f_oneway(asian, black, hispanic, other, white) F_onewayResult(statistic=np.float64(1.7744689357329695), pvalue=np.float64(0.13173183201930463))  The test output yields an F-statistic of 1.774 and a p-value of 0.1317, indicating that there is no significant difference between the means of each group. Another way to carry out an ANOVA test is to use the statsmodels library, which allows you to specify a model with a formula syntax that mirrors that used by the R programming language. R users may find this method more familiar:  import   statsmodels.api   as   sm from   statsmodels.formula.api   import   ols model   =   ols( 'age ~ race' ,   # Model formula data   =   voter_frame).fit() anova_result   =   sm.stats.anova_lm(model, typ = 2 ) print   (anova_result)

--- End of Page 2 ---

sum_sq     df         F    PR(>F) race        199.369    4.0  1.774469  0.131732 Residual  27948.102  995.0       NaN       NaN  As you can see, the statsmodels method produced the same F statistic and P-value (listed as PR(<F)) as the stats.f_oneway method. Now let’s make new age data where the group means do differ and run a second ANOVA:  np.random.seed( 12 ) # Generate random data voter_race   =   np.random.choice(a =   races, p   =   [ 0.05 ,   0.15   , 0.25 ,   0.05 ,   0.5 ], size = 1000 ) # Use a different distribution for white ages white_ages   =   stats.poisson.rvs(loc = 18 , mu = 32 , size = 1000 ) voter_age   =   stats.poisson.rvs(loc = 18 , mu = 30 , size = 1000 ) voter_age   =   np.where(voter_race == "white" , white_ages, voter_age) # Group age data by race voter_frame   =   pd.DataFrame({ "race" :voter_race, "age" :voter_age}) groups   =   voter_frame.groupby( "race" ).groups # Extract individual groups asian   =   voter_age[groups[ "asian" ]] black   =   voter_age[groups[ "black" ]] hispanic   =   voter_age[groups[ "hispanic" ]] other   =   voter_age[groups[ "other" ]] white   =   voter_age[groups[ "white" ]] # Perform the ANOVA stats.f_oneway(asian, black, hispanic, other, white) F_onewayResult(statistic=np.float64(10.164699828386366), pvalue=np.float64(4.5613242113994585e-08)) # Alternate method model   =   ols( 'age ~ race' ,   # Model formula data   =   voter_frame).fit() anova_result   =   sm.stats.anova_lm(model, typ = 2 )

--- End of Page 3 ---

print   (anova_result) sum_sq     df        F        PR(>F) race       1284.123213    4.0  10.1647  4.561324e-08 Residual  31424.995787  995.0      NaN           NaN  The test result suggests the groups don’t have the same sample means in this case, since the p-value is significant at a 99% confidence level. We know that it is the white voters who differ because we set it up that way in the code, but when testing real data, you may not know which group(s) caused the test to throw a positive result. To check which groups differ after getting a positive ANOVA result, you can perform a follow up test or “post-hoc test”. One post-hoc test is to perform a separate t-test for each pair of groups. You can perform a t-test between all pairs using by running each pair through the stats.ttest_ind() we covered in the lesson on t-tests:  # Get all race pairs race_pairs   =   [] for   race1   in   range ( 4 ): for   race2   in   range (race1 + 1 , 5 ): race_pairs.append((races[race1], races[race2])) # Conduct t-test on each pair for   race1, race2   in   race_pairs: print (race1, race2) print (stats.ttest_ind(voter_age[groups[race1]], voter_age[groups[race2]])) asian black TtestResult(statistic=np.float64(0.8386446909747979), pvalue=np.float64(0.4027281369339345), df=np.float64(189.0)) asian hispanic TtestResult(statistic=np.float64(-0.42594691924932293), pvalue=np.float64(0.6704669004240726), df=np.float64(286.0)) asian other TtestResult(statistic=np.float64(0.9795284739636), pvalue=np.float64(0.3298877500095151), df=np.float64(92.0)) asian white TtestResult(statistic=np.float64(-2.318108811252288), pvalue=np.float64(0.020804701566400217), df=np.float64(557.0)) black hispanic TtestResult(statistic=np.float64(-1.9527839210712925), pvalue=np.float64(0.05156197171952594), df=np.float64(389.0)) black other TtestResult(statistic=np.float64(0.28025754367057176), pvalue=np.float64(0.7795770111117659), df=np.float64(195.0)) black white TtestResult(statistic=np.float64(-5.379303881281834), pvalue=np.float64(1.0394212166624012e-07), df=np.float64(660.0)) hispanic other TtestResult(statistic=np.float64(1.5853626170340225), pvalue=np.float64(0.11396630528484336), df=np.float64(292.0)) hispanic white TtestResult(statistic=np.float64(-3.5160312714115376), pvalue=np.float64(0.0004641298649066684), df=np.float64(757.0)) other white

--- End of Page 4 ---

TtestResult(statistic=np.float64(-3.763809322077872), pvalue=np.float64(0.00018490576317593065), df=np.float64(563.0))  The p-values for each pairwise t-test suggest mean of white voters is likely different from the other groups, since the p-values for each t-test involving the white group is below 0.05. Using unadjusted pairwise t-tests can overestimate significance, however, because the more comparisons you make, the more likely you are to come across an unlikely result due to chance. We can adjust for this multiple comparison problem by dividing the statistical significance level by the number of comparisons made. In this case, if we were looking for a significance level of 5%, we’d be looking for p-values of 0.05/10 = 0.005 or less. This simple adjustment for multiple comparisons is known as the Bonferroni correction. The Bonferroni correction is a conservative approach to account for the multiple comparisons problem that may end up rejecting results that are actually significant. Another common post hoc-test is Tukey’s test. You can carry out Tukey’s test using the pairwise_tukeyhsd() function in the statsmodels.stats.multicomp library:  from   statsmodels.stats.multicomp   import   pairwise_tukeyhsd tukey   =   pairwise_tukeyhsd(endog = voter_age,   # Data groups = voter_race,   # Groups alpha = 0.05 )   # Significance level tukey.plot_simultaneous()   # Plot group confidence intervals plt.vlines(x = 49.57 ,ymin =- 0.5 ,ymax = 4.5 , color = "red" ) tukey.summary()   # See test summary  Multiple Comparison of Means - Tukey HSD, FWER=0.05  group1   group2   meandiff   p-adj   lower   upper   reject  asian   black   -0.8032   0.9208 -3.4423 1.836   False asian   hispanic 0.4143   0.9915 -2.1011 2.9297 False asian   other   -1.0645   0.8906 -4.2391 2.11   False asian   white   1.9547   0.1751 -0.4575 4.3668 False black   hispanic 1.2175   0.2318 -0.386   2.821   False black   other   -0.2614   0.9986 -2.7757 2.253   False black   white   2.7579   0.0   1.3217   4.194   True hispanic other   -1.4789   0.4374 -3.863   0.9053 False hispanic white   1.5404   0.004   0.3468   2.734   True other   white   3.0192   0.0028 0.7443   5.2941 True

--- End of Page 5 ---

png The output of the Tukey test shows the average difference, a confidence interval as well as whether you should reject the null hypothesis for each pair of groups at the given significance level. In this case, the test suggests we reject the null hypothesis for 3 pairs, with each pair including the “white” category. This suggests the white group is likely different from the others. The 95% confidence interval plot reinforces the results visually: only 1 other group’s confidence interval overlaps the white group’s confidence interval.  Wrap Up  The ANOVA test lets us check whether a numeric response variable varies according to the levels of a categorical variable. Python’s scipy library makes it easy to perform an ANOVA without diving too deep into the details of the procedure. Next time, we’ll move on from statistical inference to the final topic of this guide: predictive modeling.

--- End of Page 6 ---

Q1.Explain input and output characteristics for f_oneway function from stats library   Ans. The scipy.stats.f_oneway function is used to perform a one-way ANOVA test. Here are its input and output characteristics: Inputs: It takes two or more array-like objects as positional arguments, where each array represents the sample observations for a different group. For example, stats.f_oneway(group1_data, group2_data, group3_data, ...). Each array should contain numerical data. Outputs: It returns a F_onewayResult object, which is a named tuple containing two main values: F-statistic (statistic): This is the calculated F-value from the ANOVA test. A larger F-statistic typically indicates a greater difference between the group means relative to the variability within the groups. P-value (pvalue): This is the probability of observing an F-statistic as extreme as, or more extreme than, the one calculated, assuming the null hypothesis is true (i.e., that all group means are equal). A small p-value (typically less than 0.05) suggests that there is a statistically significant difference between at least two of the group means.  Q2.Explain results of sm.stats.anova_lm fucntion.   Ans. The output of the sm.stats.anova_lm function, as shown in your anova_result DataFrame, presents the results of the ANOVA test in a tabular format. Let’s break down each column: sum_sq (Sum of Squares): This column represents the variability in the data that can be attributed to different sources. In your output: race: This is the Sum of Squares for the ‘race’ factor (between-group variability). It quantifies how much the group means differ from the overall mean. Residual: This is the Sum of Squares for the residuals (within-group variability or error). It quantifies the variability within each group that is not explained by the ‘race’ factor. df (Degrees of Freedom): This indicates the number of independent pieces of information used to calculate the sum of squares. race: The degrees of freedom for the ‘race’ factor. It’s calculated as (number of groups - 1). Residual: The degrees of freedom for the residuals. It’s calculated as (total number of observations - number of groups). F (F-statistic): This is the test statistic for the ANOVA. It’s calculated as the ratio of the mean square for the factor (sum_sq / df) to the mean square for the residuals (sum_sq / df). A larger F- statistic suggests that the variability between group means is greater than the variability within groups. PR(>F) (P-value): This is the probability of observing an F-statistic as extreme as, or more extreme than, the one calculated, assuming the null hypothesis is true (i.e., that all group means are equal). A small p-value (typically less than your chosen significance level, e.g., 0.05) indicates that you should reject the null hypothesis, suggesting that there is a statistically significant difference between at least two of the group means.

--- End of Page 7 ---

Q3.What role ANOVA plays in data analytics?   Ans. ANOVA (Analysis of Variance) plays a crucial role in data analytics, particularly in statistical hypothesis testing and experimental design. Here’s a breakdown of its main roles: Comparing Multiple Group Means: Its primary role is to determine if there are any statistically significant differences between the means of three or more independent groups. While t-tests compare two means, ANOVA extends this to multiple groups, which is essential when you have a categorical variable with several levels. Identifying Significant Factors: In experiments or observational studies, ANOVA helps identify which categorical factors (or independent variables) have a significant impact on a continuous outcome variable (dependent variable). For example, it can tell you if different teaching methods (factor) lead to different test scores (outcome). Controlling for Type I Error Rate: If you were to perform multiple pairwise t-tests to compare all combinations of groups, the probability of making a Type I error (false positive) would increase significantly. ANOVA provides a single, overall test to see if any group means differ, thereby controlling the family-wise error rate. Foundation for Post-Hoc Tests: When an ANOVA yields a significant result (meaning at least one group mean is different), it doesn’t tell you which specific groups differ. This is where post-hoc tests (like Tukey’s HSD or Bonferroni correction) come in. ANOVA serves as the necessary precursor to these follow-up analyses, guiding further investigation. Understanding Variability: ANOVA works by partitioning the total variability in the data into different components: variability between groups and variability within groups. By comparing these variances, it assesses whether the differences observed between group means are likely due to the factor being studied or just random chance. Applications in Various Fields: ANOVA is widely used in fields such as: Marketing: Comparing the effectiveness of different advertising campaigns on sales. Medicine: Assessing the impact of various drug dosages on patient outcomes. Agriculture: Determining if different fertilizers affect crop yield. Social Sciences: Studying if different demographic groups have different average scores on a psychological test. In essence, ANOVA is a powerful tool for making informed decisions and drawing valid conclusions about group differences from data, which is fundamental to many data analytics tasks.  Q4.When to use one way ANOVA? Explain with real world case.   Ans. You should use a one-way ANOVA when you want to compare the means of a continuous dependent variable across three or more independent (categorical) groups. The ‘one-way’ refers to having only one independent variable (factor) with multiple levels or categories.

--- End of Page 8 ---

Here are the key conditions for using one-way ANOVA: One Continuous Dependent Variable: The variable you are measuring (the outcome) must be continuous (e.g., age, height, test scores, sales figures). One Categorical Independent Variable (Factor): You have one grouping variable that divides your data into three or more distinct, independent groups (e.g., different drug treatments, different teaching methods, different brands). Independence of Observations: The observations within each group, and between the groups, must be independent of each other. Normal Distribution: The dependent variable should be approximately normally distributed within each group. Homogeneity of Variances: The variances of the dependent variable should be roughly equal across all groups. Real-World Case: Scenario: A pharmaceutical company wants to test the effectiveness of three different dosages of a new drug (10mg, 20mg, and 30mg) on reducing blood pressure. They also include a placebo group. Dependent Variable: Blood Pressure Reduction (measured in mmHg, a continuous variable). Independent Variable (Factor): Drug Dosage (categorical with four levels: Placebo, 10mg, 20mg, 30mg). Why use one-way ANOVA here? You have more than two groups (Placebo, 10mg, 20mg, 30mg). If you only had two groups (e.g., Placebo vs.   10mg), a t-test would suffice. But with four groups, performing multiple t-tests would increase the chance of making a Type I error (false positive). ANOVA will tell you if there is a statistically significant difference in mean blood pressure reduction among any of the dosage groups. If the ANOVA is significant, you would then use post-hoc tests (like Tukey’s HSD) to find out which specific dosage groups differ from each other.  Q5.When to use 2 way ANOVA ? Explain with real world case.   Ans. You should use a two-way ANOVA when you want to compare the means of a continuous dependent variable across two independent categorical variables (factors). It allows you to examine the main effect of each independent variable, as well as the interaction effect between them. Here are the key conditions for using two-way ANOVA: One Continuous Dependent Variable: The variable you are measuring (the outcome) must be continuous (e.g., test scores, crop yield, sales volume). Two Categorical Independent Variables (Factors): You have two grouping variables, each with two or more levels or categories. Independence of Observations: The observations within each group, and between the groups, must be independent of each other. Normal Distribution: The dependent variable should be approximately normally distributed within each group. Homogeneity of Variances: The variances of the dependent variable should be roughly equal across all groups. Real-World Case: Scenario: A company wants to evaluate the effectiveness of two different marketing strategies (A and B) on sales, and also consider if the region (North, South, East, West) where the strategy is implemented has an effect, or if the effectiveness of a strategy depends on the region. Dependent Variable: Sales Volume (a continuous variable). Independent Variable 1

--- End of Page 9 ---

(Factor 1): Marketing Strategy (categorical with two levels: A, B). Independent Variable 2 (Factor 2): Region (categorical with four levels: North, South, East, West). Why use two-way ANOVA here? You have two independent categorical variables (Marketing Strategy and Region) and you want to see their individual effects on Sales Volume, as well as if there’s an interaction between them. A two-way ANOVA can tell you: Main effect of Marketing Strategy: Is there a significant difference in sales between Strategy A and Strategy B, regardless of region? Main effect of Region: Is there a significant difference in sales across the different regions, regardless of the marketing strategy used? Interaction effect (Marketing Strategy x Region): Does the effect of the marketing strategy on sales depend on the region? For example, perhaps Strategy A works much better in the North, while Strategy B performs exceptionally well in the South. If an interaction is significant, it means you can’t simply interpret the main effects independently; you need to consider how the two factors combine to influence the outcome.